# Data exploration

This notebook is used in order to do exploratory data analysis. The figures and data shown in the first part of the methodology is gathered here.
There is three main parts to this notebook. First, the spacing and dimensions of the raw mri data is gathered. Secondly, the data is resampled and data after resampling is gathered. Thirdly, data of the class distribution is gathered. Lastly, I check if there are any empty masks and if so where they are.
Most figures of import are written to be saved inside of a folder called **figures**, so the program might crash without it.

In [ ]:
# imports
from pathlib import Path
from typing import Union
from collections import Counter
import numpy as np
import SimpleITK as sitk

# plotting
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
from matplotlib.colors import ListedColormap, BoundaryNorm

In [ ]:
# Settings for matplotlib and seaborn to make them look nice
mpl.rcParams.update({
    "font.size": 50,
    "axes.titlesize": 13,
    "axes.labelsize": 100,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "figure.figsize": (4.5, 3.2),

    # PGF / LaTeX settings
    "pgf.texsystem": "pdflatex",
    "pgf.rcfonts": False,

    # avoid requiring full LaTeX rendering for every text element
    "text.usetex": False,
})

sns.set_theme(style="ticks", context="paper")

## 1. Pre resampling data

In [ ]:
def read_config(filepath: Union[str, Path]) -> dict:
    schema = {
        "ED": int,
        "ES": int,
        "Group": str,
        "Height": float,
        "NbFrame": int,
        "Weight": float,
    }
    config = {}
    with open(Path(filepath) / "Info.cfg") as f:
        for line in f:
            if not line.strip():
                continue
            key, value = line.split(":", 1)
            key, value = key.strip(), value.strip()
            config[key] = schema.get(key, str)(value)
    return config


def iterate_patients(base_path: Union[str, Path]):
    for i in range(1, 101):
        patient_name = f"patient{i:03d}"
        path_to_patient = Path(base_path) / patient_name
        conf = read_config(path_to_patient)

        paths = {}
        for phase in ["ED", "ES"]:
            frame = f"frame{conf[phase]:02d}"
            paths[phase] = {
                "img": path_to_patient / f"{patient_name}_{frame}.nii.gz",
                "gt":  path_to_patient / f"{patient_name}_{frame}_gt.nii.gz",
            }

        yield patient_name, conf, paths

BAR_COLOR = "#4C72B0"

def plot_hist(ax, data, xlabel, bins=10):
    data = np.array(data)

    sns.histplot(
        data=data,
        bins=bins,
        color=BAR_COLOR,
        edgecolor="black",
        ax=ax
    )

    for patch in ax.patches:
        patch.set_facecolor(BAR_COLOR)

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Number of scans")
    ax.grid(axis='y', alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


def print_stats(name, data):
    data = np.array(data)

    d_min = np.min(data)
    d_max = np.max(data)
    d_mean = np.mean(data)
    d_median = np.median(data)

    print(f"{name}")
    print(f"  Min:    {d_min}")
    print(f"  Max:    {d_max}")
    print(f"  Mean:   {d_mean:.2f}")
    print(f"  Median: {d_median}")
    print()

In [ ]:
# Data collection from the training set
X = []
Y = []
Z = []
dX = []
dY = []
dZ = []

# train set
for patient_name, conf, paths in iterate_patients("data/ACDC/database/training"):
    ed_img = sitk.ReadImage(paths["ED"]["img"])
    es_img = sitk.ReadImage(paths["ES"]["img"])

    x, y, z = ed_img.GetSize()
    spacing = ed_img.GetSpacing()

    # sanity checks to ensure that ED and ES images have the same shape and spacing
    assert ed_img.GetSize() == es_img.GetSize(), f"Shape mismatch for patient {patient_name}"
    assert spacing == es_img.GetSpacing(), f"Spacing mismatch for patient {patient_name}"

    X.append(x)
    Y.append(y)
    Z.append(z)
    dX.append(spacing[0])
    dY.append(spacing[1])
    dZ.append(spacing[2])

print(f"The total amount of slices in the training are {sum(Z)*2}") # *2 becaause we have both ED and ES phases

In [ ]:
# dimesnions
fig, ax = plt.subplots(figsize=(4, 3.5))
plot_hist(ax, X, "X dimension (pixels)", bins=10)
fig.tight_layout()
fig.savefig("figures/dataset_dim_hist_X.pdf", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(4, 3.5))
plot_hist(ax, Y, "Y dimension (pixels)", bins=10)
fig.tight_layout()
fig.savefig("figures/dataset_dim_hist_Y.pdf", bbox_inches="tight")
plt.show()

bins = np.arange(min(Z), max(Z) + 2) - 0.5
fig, ax = plt.subplots(figsize=(4, 3.5))
plot_hist(ax, Z, "Number of slices", bins=bins)
fig.tight_layout()
fig.savefig("figures/dataset_dim_hist_Z.pdf", bbox_inches="tight")
plt.show()

print_stats("X dimension (pixels)", X)
print_stats("Y dimension (pixels)", Y)
print_stats("Number of slices", Z)

In [ ]:
# Spacing
fig, ax = plt.subplots(figsize=(4, 3.5))
plot_hist(ax, dX, "X spacing (mm)", bins=10)
fig.tight_layout()
fig.savefig("figures/dataset_spacing_hist_X.pdf", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(4, 3.5))
plot_hist(ax, dY, "Y spacing (mm)", bins=10)
fig.tight_layout()
fig.savefig("figures/dataset_spacing_hist_Y.pdf", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(4, 3.5))
plot_hist(ax, dZ, "Z spacing (mm)", bins=10)
fig.tight_layout()
fig.savefig("figures/dataset_spacing_hist_Z.pdf", bbox_inches="tight")
plt.show()


print_stats("X spacing (mm)", dX)
print_stats("Y spacing (mm)", dY)
print_stats("Z spacing (mm)", dZ)

## 2. Resampling data

In [ ]:
# functions
sitk.ProcessObject_SetGlobalWarningDisplay(False)

def resample_xy_to_mm(nifti_path, is_label=False, spacing=1.0, default_value=0):
    img = sitk.ReadImage(nifti_path)

    original_spacing = img.GetSpacing()  # (sx, sy, sz)
    original_size    = img.GetSize()     # (x, y, z)

    new_spacing = (float(spacing), float(spacing), original_spacing[2])

    new_size = [
        int(round(original_size[0] * (original_spacing[0] / new_spacing[0]))),
        int(round(original_size[1] * (original_spacing[1] / new_spacing[1]))),
        original_size[2],
    ]
    new_size = [max(1, s) for s in new_size]

    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(img)         
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetSize(new_size)
    resampler.SetTransform(sitk.Transform())
    resampler.SetDefaultPixelValue(default_value)

    if is_label:
        resampler.SetInterpolator(sitk.sitkNearestNeighbor)
        img = sitk.Cast(img, sitk.sitkUInt8)
    else:
        resampler.SetInterpolator(sitk.sitkLinear)

    return resampler.Execute(img)

def sitk_to_numpy(img):
    return sitk.GetArrayFromImage(img)  # (z, y, x)

def get_slice_positions_z(img):
    size = img.GetSize()        # (x, y, z)
    z_positions = []

    for k in range(size[2]):
        # index -> physical point
        x, y, z = img.TransformIndexToPhysicalPoint((0, 0, k))
        z_positions.append(z)

    return np.array(z_positions)

def plot_slice_overlay(img, msk, spacing=None, z=None, save_path=None, alpha=0.4, figsize=(5, 5), show=True):
    if z is None:
        z = img.shape[0] // 2

    img_slice = img[z]
    msk_slice = msk[z]

    p1, p99 = np.percentile(img_slice, (1, 99))
    img_disp = np.clip(img_slice, p1, p99)

    cmap = ListedColormap([
        (0, 0, 0, 0),
        (0, 1, 0, 1),
        (1, 0, 0, 1),
        (0, 0, 1, 1),
    ])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)
    msk_masked = np.ma.masked_where(msk_slice == 0, msk_slice)

    if spacing is not None:
        print(f"spacing: ({spacing[0]:.4f}, {spacing[1]:.4f}, {spacing[2]:.4f}) mm")
    print(f"slice: {z}")

    if save_path:
        stem = str(save_path)
        dot = stem.rfind(".")
        if dot != -1 and "." in stem.split("/")[-1]:
            img_save = stem[:dot] + "_image" + stem[dot:]
            ov_save  = stem[:dot] + "_overlay" + stem[dot:]
        else:
            img_save = stem + "_image"
            ov_save  = stem + "_overlay"
    else:
        img_save = None
        ov_save  = None

    def _save_tight(fig, path):
        """Save with zero padding/whitespace."""
        fig.savefig(path, bbox_inches="tight", pad_inches=0, dpi=300)

    # --- plain image ---
    fig_img, ax_img = plt.subplots(figsize=figsize)
    fig_img.subplots_adjust(left=0, right=1, top=1, bottom=0)
    ax_img.imshow(img_disp, cmap="gray")
    ax_img.axis("off")
    if img_save:
        _save_tight(fig_img, img_save)
    if show:
        plt.show()
    plt.close(fig_img)

    # --- overlay (no legend) ---
    fig_ov, ax_ov = plt.subplots(figsize=figsize)
    fig_ov.subplots_adjust(left=0, right=1, top=1, bottom=0)
    ax_ov.imshow(img_disp, cmap="gray")
    ax_ov.imshow(msk_masked, cmap=cmap, norm=norm, alpha=alpha, interpolation="nearest")
    ax_ov.axis("off")
    if ov_save:
        _save_tight(fig_ov, ov_save)
    if show:
        plt.show()
    plt.close(fig_ov)

### Illustrative example of pre- vs post resampling

In [ ]:
# Before spacing
patientid = 57
frame = 1
img_path = f"data/ACDC/database/training/patient{patientid:03d}/patient{patientid:03d}_frame{frame:02d}.nii.gz"
msk_path = f"data/ACDC/database/training/patient{patientid:03d}/patient{patientid:03d}_frame{frame:02d}_gt.nii.gz"

img_sitk = sitk.ReadImage(img_path)
msk_sitk = sitk.ReadImage(msk_path)

img = sitk.GetArrayFromImage(img_sitk)
msk = sitk.GetArrayFromImage(msk_sitk)

print(f"Patient: {patientid}")
print(f"Frame: {frame}")
print(f"img dimensions: {img.shape}")

plot_slice_overlay(
    img,
    msk,
    spacing=img_sitk.GetSpacing(),
    alpha=0.65,
    save_path="figures/example_original.pdf"
)

In [ ]:
# after resampling
img_sitk = resample_xy_to_mm(img_path, is_label=False, spacing=1.6)
msk_sitk = resample_xy_to_mm(msk_path, is_label=True, spacing=1.6)

img = sitk.GetArrayFromImage(img_sitk)
msk = sitk.GetArrayFromImage(msk_sitk)


print(f"Patient: {patientid}")
print(f"Frame: {frame}")
print(f"img dimensions: {img.shape}")
plot_slice_overlay(
    img,
    msk,
    spacing=img_sitk.GetSpacing(),
    alpha=0.65,
    save_path="figures/example_resampled.pdf"
)

### Stats

In [ ]:
X = []
Y = []
Z = []
dX = []
dY = []
dZ = []
# train set
for patient_name, conf, paths in iterate_patients("data/ACDC/database/training"):
    ed_img = resample_xy_to_mm(paths["ED"]["img"], spacing=1.6)
    es_img = resample_xy_to_mm(paths["ES"]["img"], spacing=1.6)

    x, y, z = ed_img.GetSize()
    spacing = ed_img.GetSpacing()

    # --- size & spacing consistency ---
    assert ed_img.GetSize() == es_img.GetSize(), f"Shape mismatch for patient {patient_name}"
    assert spacing == es_img.GetSpacing(), f"Spacing mismatch for patient {patient_name}"

    # --- z position consistency ---
    z_positions = get_slice_positions_z(ed_img)
    assert np.allclose(z_positions, get_slice_positions_z(es_img)), f"Z positions mismatch for patient {patient_name}"
    assert np.allclose(z_positions, np.arange(z) * spacing[2] + ed_img.GetOrigin()[2]), f"Z positions do not match expected values for patient {patient_name}"

    # --- uniform slice spacing ---
    dz_phys = np.diff(z_positions)
    assert np.allclose(dz_phys, dz_phys[0]), f"Uneven slice spacing for patient {patient_name}"

    X.append(x)
    Y.append(y)
    Z.append(z)
    dX.append(spacing[0])
    dY.append(spacing[1])
    dZ.append(spacing[2])

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.5))
plot_hist(ax, X, "X dimension (pixels)", bins=10)
fig.tight_layout()
fig.savefig("figures/dataset_dim_resampled_hist_X.pdf", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(4, 3.5))
plot_hist(ax, Y, "Y dimension (pixels)", bins=10)
fig.tight_layout()
fig.savefig("figures/dataset_dim_resampled_hist_Y.pdf", bbox_inches="tight")
plt.show()

bins = np.arange(min(Z), max(Z) + 2) - 0.5
fig, ax = plt.subplots(figsize=(4, 3.5))
plot_hist(ax, Z, "Number of slices", bins=bins)
fig.tight_layout()
fig.savefig("figures/dataset_dim_resampled_hist_Z.pdf", bbox_inches="tight")
plt.show()

print_stats("X dimension (pixels)", X)
print_stats("Y dimension (pixels)", Y)
print_stats("Number of slices", Z)


In [ ]:
# just a quick check to see if resampling worked as expected, should be more consistent spacing now
plt.hist(dX, bins=20)
plt.title("Distribution of X spacing in MRI scans")
plt.show()

plt.hist(dY, bins=20)
plt.title("Distribution of Y spacing in MRI scans")
plt.show()

plt.hist(dZ, bins=20)
plt.title("Distribution of Z spacing in MRI scans")
plt.show()

## 3. Class distribution

In [ ]:
class_counter = Counter()
pathology_counter = Counter()
patient_list = []
y = []

# train set masks
for patient_name, conf, paths in iterate_patients("data/ACDC/database/training"):
    pathology_counter[conf["Group"]] += 1
    patient_list.append(patient_name)
    y.append(conf["Group"])

    for phase in ["ED", "ES"]:
        gt = sitk.GetArrayFromImage(sitk.ReadImage(str(paths[phase]["gt"]))).astype(np.int32)
        labels, counts = np.unique(gt, return_counts=True)
        for lbl, cnt in zip(labels, counts):
            class_counter[int(lbl)] += int(cnt)

# print results
print("Class voxel counts (ED + ES combined):")
for cls in sorted(class_counter):
    print(f"Class {cls}: {class_counter[cls]}")

total = sum(class_counter.values())
print("\nClass voxel percentages:")
for cls in sorted(class_counter):
    pct = 100 * class_counter[cls] / total
    print(f"Class {cls}: {pct:.2f}%")

print("\nPathology group counts:")
for group, cnt in pathology_counter.items():
    print(f"{group}: {cnt}")

## 4. Checking if masks are empty

In [ ]:
empty_slices_total = 0
total_slices_total = 0

empty_slices_per_phase = Counter()
total_slices_per_phase = Counter()

empty_volumes_total = 0
empty_volumes_per_phase = Counter()

# classify empty slices as first / last / middle / only
empty_location_type = {
    "ED": Counter(),
    "ES": Counter(),
}

for patient_name, conf, paths in iterate_patients("data/ACDC/database/training"):
    for phase in ["ED", "ES"]:
        gt_resampled = resample_xy_to_mm(
            paths[phase]["gt"], is_label=True, spacing=1.6, default_value=0
        )
        gt_np = sitk_to_numpy(gt_resampled)  # shape: (z, y, x)

        z = gt_np.shape[0]
        total_slices_total += z
        total_slices_per_phase[phase] += z

        # check if entire volume is empty
        if np.all(gt_np == 0):
            empty_volumes_total += 1
            empty_volumes_per_phase[phase] += 1

        # slice is empty if all zeros in (y, x)
        empty_per_slice = np.all(gt_np == 0, axis=(1, 2))  # shape: (z,)
        empty_indices = np.where(empty_per_slice)[0]
        n_empty = len(empty_indices)

        empty_slices_total += n_empty
        empty_slices_per_phase[phase] += n_empty

        for idx in empty_indices:
            if z == 1:
                empty_location_type[phase]["only"] += 1
            elif idx == 0:
                empty_location_type[phase]["first"] += 1
            elif idx == z - 1:
                empty_location_type[phase]["last"] += 1
            else:
                empty_location_type[phase]["middle"] += 1

print(f"Empty volumes (total): {empty_volumes_total} / 200")
print("Empty volumes per phase:")
for ph in ["ED", "ES"]:
    print(f"  {ph}: {empty_volumes_per_phase[ph]} / 100")

print(f"\nEmpty slices (total): {empty_slices_total} / {total_slices_total}")
print("Empty slices per phase:")
for ph in ["ED", "ES"]:
    print(f"  {ph}: {empty_slices_per_phase[ph]} / {total_slices_per_phase[ph]}")

for ph in ["ED", "ES"]:
    print(f"\n=== {ph} empty-slice location summary ===")

    print("Type counts:")
    total_empty_here = sum(empty_location_type[ph].values())
    for k in ["first", "last", "middle", "only"]:
        v = empty_location_type[ph][k]
        if total_empty_here > 0:
            print(f"  {k}: {v} ({100*v/total_empty_here:.1f}%)")
        else:
            print(f"  {k}: {v}")